> ⚠️ **NOTEBOOK SUPERSEDED — à lire avec précaution.**
>
> Ce notebook compare les alertes du pipeline aux scores SLS de **Fall 2019**, qui sont
> **non synchrones** avec les données capteurs (décalage de ~8 mois : SLS de janvier-mars 2019,
> IceTag de novembre-décembre 2019). La comparaison clinique présentée ici n'est donc **pas valide**.
>
> La version correcte de l'évaluation clinique utilise **Winter 2019** (scores SLS synchrones,
> mars 2019) et se trouve dans le **notebook 06 (diagnostic de séparabilité)**.
> L'application propre du pipeline aux 4 saisons est dans le **notebook 05**.
> La concordance comportementale (Tâche 1.2 du SOW) est dans le **notebook 10**.
>
> Ce notebook est conservé pour traçabilité uniquement.

---

# Exécution du pipeline personnel sur les données McGill

**Objectif** : appliquer le pipeline `IF + règles métier` avec les **paramètres gelés** du mémoire sur les données McGill converties (`mcgill_brut.csv`), puis comparer les alertes obtenues aux **scores cliniques SLS historiques**.

**Important** : les labels SLS datent de **janvier-mars 2019**, alors que les données IceTag Fall 2019 couvrent **novembre-décembre 2019**. Il s'agit donc d'une **validation exploratoire externe non synchrone**, pas d'une validation clinique stricte.


## 1. Imports et chemins

Le notebook suppose que `02_convert_icetag_to_pipeline.ipynb` a déjà été exécuté et que le fichier `mcgill_brut.csv` existe.


In [19]:
import json
import sys
from pathlib import Path
import inspect

import pandas as pd

# Racine du projet et du dossier McGill
PROJECT_ROOT = Path('..').resolve().parent
MCGILL_ROOT = Path('..').resolve()
REPORTS_DIR = MCGILL_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

# Ajouter le projet au PYTHONPATH
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.io import load_csv, COW, TIME
from core.pipeline import run_pipeline_herd
from scripts.validation_common import normalize_params

MCGILL_CSV = MCGILL_ROOT / 'mcgill_brut.csv'
SLS_CSV = MCGILL_ROOT / 'mcgill_sls_labels.csv'
FROZEN_PARAMS_JSON = PROJECT_ROOT / 'data' / 'final_thresholds_v1.json'

SUMMARY_OUT = REPORTS_DIR / 'mcgill_pipeline_summary.csv'
PRED_OUT = REPORTS_DIR / 'mcgill_pipeline_predictions.csv'
DAILY_OUT = REPORTS_DIR / 'mcgill_pipeline_daily_alerts.csv'
COMPARE_OUT = REPORTS_DIR / 'mcgill_pipeline_vs_sls.csv'

assert MCGILL_CSV.exists(), f'Fichier manquant: {MCGILL_CSV}'
assert SLS_CSV.exists(), f'Fichier manquant: {SLS_CSV}'
assert FROZEN_PARAMS_JSON.exists(), f'Fichier manquant: {FROZEN_PARAMS_JSON}'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('MCGILL_ROOT  =', MCGILL_ROOT)
print('MCGILL_CSV   =', MCGILL_CSV)
print('SLS_CSV      =', SLS_CSV)
print('PARAMS_JSON  =', FROZEN_PARAMS_JSON)
print('REPORTS_DIR  =', REPORTS_DIR)


PROJECT_ROOT = /Users/alioubarry/PROJECT
MCGILL_ROOT  = /Users/alioubarry/PROJECT/mcgill_iot_cattle
MCGILL_CSV   = /Users/alioubarry/PROJECT/mcgill_iot_cattle/mcgill_brut.csv
SLS_CSV      = /Users/alioubarry/PROJECT/mcgill_iot_cattle/mcgill_sls_labels.csv
PARAMS_JSON  = /Users/alioubarry/PROJECT/data/final_thresholds_v1.json
REPORTS_DIR  = /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports


## 2. Charger les paramètres gelés du mémoire

On relit les paramètres officiels figés dans `data/final_thresholds_v1.json`, puis on les normalise avec les helpers du projet.


In [22]:
with open(FROZEN_PARAMS_JSON, 'r', encoding='utf-8') as f:
    frozen = json.load(f)

base_params = normalize_params(frozen['pipeline_defaults'])

# Le JSON gelé peut contenir des clés de documentation ou de config interne
# non exposées dans la signature publique de run_pipeline_herd().
allowed = set(inspect.signature(run_pipeline_herd).parameters.keys())
base_params = {k: v for k, v in base_params.items() if k in allowed}

print('Paramètres effectivement transmis au pipeline :')
base_params


Paramètres effectivement transmis au pipeline :


{'interval': '15T',
 'window_baseline': 24,
 'contamination': 0.06,
 'baseline_ratio': 0.6,
 'random_state': 42,
 'persist_hours': 7,
 'alert_min': 2,
 'mix_mode': 'MIX',
 'mix_rate_thr': 0.24,
 'z_low_thr': -2.0,
 'z_high_thr': 2.0,
 'cooldown_hours': 12,
 'mi_z_high_thr': 2.2,
 'coverage_min_pct': 25.0}

## 3. Charger les données McGill converties et les labels SLS


In [25]:
df_raw = load_csv(str(MCGILL_CSV))
labels_df = pd.read_csv(SLS_CSV)
labels_df[COW] = labels_df[COW].astype(str)

print('Données capteurs:')
print(df_raw.shape)
print(df_raw[[COW, TIME]].head().to_string(index=False))
print()
print('Labels cliniques:')
print(labels_df.shape)
labels_df.head(10)


Données capteurs:
(93187, 10)
 Cow                   T
2041 2019-11-11 18:45:00
2041 2019-11-11 19:00:00
2041 2019-11-11 19:15:00
2041 2019-11-11 19:30:00
2041 2019-11-11 19:45:00

Labels cliniques:
(30, 8)


,Cow,SLS_Baseline_Jan2019,SLS_Midway_Mar2019,Statut,Bins_15min,Jours,Debut,Fin
0,821,2.0,2.0,boiteuse_persistante,3145,34,2019-11-11,2019-12-14
1,2041,NaN,NaN,pas_de_score,3145,34,2019-11-11,2019-12-14
2,2057,1.0,2.0,legere_aggravee,3144,34,2019-11-11,2019-12-14
3,2062,NaN,NaN,pas_de_score,3144,34,2019-11-11,2019-12-14
4,2063,1.0,0.0,legere_guerie,3140,34,2019-11-11,2019-12-14
5,2066,NaN,NaN,pas_de_score,3142,34,2019-11-11,2019-12-14
6,2078,0.0,0.0,saine,3140,34,2019-11-11,2019-12-14
7,2081,1.0,NaN,legere,3141,34,2019-11-11,2019-12-14
8,3435,NaN,NaN,pas_de_score,3140,34,2019-11-11,2019-12-14
9,3437,0.0,1.0,saine_degradee,3145,34,2019-11-11,2019-12-14


## 4. Vue d'ensemble rapide du corpus McGill


In [28]:
print(f'Nombre de vaches      : {df_raw[COW].nunique()}')
print(f'Nombre de bins 15 min : {len(df_raw):,}')
print(f'Période               : {df_raw[TIME].min()} -> {df_raw[TIME].max()}')

coverage = (
    df_raw.assign(Day=df_raw[TIME].dt.floor('D'))
         .groupby(COW)
         .agg(n_bins=(TIME, 'size'), n_days=('Day', 'nunique'))
         .reset_index()
)
coverage.head(10)


Nombre de vaches      : 30
Nombre de bins 15 min : 93,187
Période               : 2019-11-11 18:45:00 -> 2019-12-14 13:00:00


,Cow,n_bins,n_days
0,2041,3145,34
1,2057,3144,34
2,2062,3144,34
3,2063,3140,34
4,2066,3142,34
5,2078,3140,34
6,2081,3141,34
7,3435,3140,34
8,3437,3145,34
9,3444,3142,34


## 5. Exécuter le pipeline gelé sur tout le troupeau McGill

Cette étape applique **exactement** le pipeline du mémoire (`IF + règles`) sur les 30 vaches McGill.


In [31]:
%%time
summary_df, pred_df = run_pipeline_herd(df_raw, **base_params)

print('summary_df:', summary_df.shape)
print('pred_df   :', pred_df.shape)
summary_df.head(10)


summary_df: (30, 11)
pred_df   : (93860, 139)
CPU times: user 23.5 s, sys: 465 ms, total: 24 s
Wall time: 25.4 s


,n_bins,if_anomaly_points,problem_points,lameness_points,problem_starts,lameness_starts,lameness_notifs,critique_points,coverage_mean,coverage_min,Cow
0,3144,193,78,78,10,10,8,17,100.0,100.0,2062
1,3072,206,92,92,8,8,8,13,100.0,100.0,8517
2,3141,172,72,72,7,7,7,16,100.0,100.0,5879
3,3142,222,34,34,7,7,6,5,100.0,100.0,2066
4,3144,182,47,47,6,6,6,7,100.0,100.0,5865
5,3144,219,74,74,7,7,5,4,100.0,100.0,5871
6,3141,177,47,47,6,6,5,7,100.0,100.0,2081
7,3145,166,56,56,6,6,5,11,100.0,100.0,2041
8,3072,232,69,69,5,5,5,7,100.0,100.0,8525
9,3072,182,34,34,5,5,5,8,100.0,100.0,8527


## 6. Vérification rapide des sorties pipeline


In [34]:
cols_of_interest = [
    COW, TIME, 'if_anomaly_point', 'pred_problem_episode', 'pred_lameness_episode',
    'pred_lameness_start', 'notif_lameness', 'alert_level',
    'if_anom_k', 'anom_rate_k', 'coherence_boiterie'
]
existing_cols = [c for c in cols_of_interest if c in pred_df.columns]
pred_df[existing_cols].head(20)


,Cow,T,if_anomaly_point,pred_problem_episode,pred_lameness_episode,pred_lameness_start,notif_lameness,alert_level,if_anom_k,anom_rate_k,coherence_boiterie
0,2041,2019-11-11 18:45:00,0,0,0,0,0,normal,0,0.000000,0
1,2041,2019-11-11 19:00:00,0,0,0,0,0,normal,0,0.000000,0
2,2041,2019-11-11 19:15:00,0,0,0,0,0,normal,0,0.000000,0
3,2041,2019-11-11 19:30:00,0,0,0,0,0,normal,0,0.000000,0
4,2041,2019-11-11 19:45:00,0,0,0,0,0,normal,0,0.000000,0
5,2041,2019-11-11 20:00:00,0,0,0,0,0,normal,0,0.000000,0
6,2041,2019-11-11 20:15:00,0,0,0,0,0,normal,0,0.000000,0
7,2041,2019-11-11 20:30:00,0,0,0,0,0,normal,0,0.000000,0
8,2041,2019-11-11 20:45:00,0,0,0,0,0,normal,0,0.000000,0
9,2041,2019-11-11 21:00:00,1,0,0,0,0,suspect,1,0.035714,0


## 7. Fusion avec les labels SLS et création de groupes cliniques

On crée ici une lecture **prudente** des labels:
- `boiteuse_stricte` : SLS >= 2 à au moins une des deux évaluations
- `saine_stricte` : SLS = 0 aux deux évaluations
- `intermediaire` : cas légers / mixtes
- `sans_score` : aucun SLS disponible


In [37]:
compare_df = summary_df.merge(labels_df, on=COW, how='left')

for c in ['SLS_Baseline_Jan2019', 'SLS_Midway_Mar2019']:
    compare_df[c] = pd.to_numeric(compare_df[c], errors='coerce')


def clinical_group(row):
    b = row['SLS_Baseline_Jan2019']
    m = row['SLS_Midway_Mar2019']
    if pd.isna(b) and pd.isna(m):
        return 'sans_score'
    vals = [v for v in [b, m] if pd.notna(v)]
    if vals and max(vals) >= 2:
        return 'boiteuse_stricte'
    if pd.notna(b) and pd.notna(m) and b == 0 and m == 0:
        return 'saine_stricte'
    return 'intermediaire'

compare_df['clinical_group'] = compare_df.apply(clinical_group, axis=1)
compare_df = compare_df.sort_values(['clinical_group', 'lameness_notifs', 'lameness_starts'], ascending=[True, False, False]).reset_index(drop=True)

compare_df[[COW, 'clinical_group', 'Statut', 'lameness_notifs', 'lameness_starts', 'lameness_points', 'if_anomaly_points', 'coverage_mean']].head(20)


,Cow,clinical_group,Statut,lameness_notifs,lameness_starts,lameness_points,if_anomaly_points,coverage_mean
0,821,boiteuse_stricte,boiteuse_persistante,4,5,35,218,100.0
1,3444,boiteuse_stricte,boiteuse_aggravee,4,5,54,204,100.0
2,2057,boiteuse_stricte,legere_aggravee,4,4,38,169,100.0
3,5874,boiteuse_stricte,boiteuse_amelioree,3,4,29,198,100.0
4,5857,boiteuse_stricte,boiteuse_amelioree,2,2,34,168,100.0
5,2081,intermediaire,legere,5,6,47,177,100.0
6,2063,intermediaire,legere_guerie,4,7,19,198,100.0
7,3437,intermediaire,saine_degradee,4,4,33,190,100.0
8,5875,intermediaire,saine_degradee,3,4,32,179,100.0
9,8500,intermediaire,saine_degradee,0,0,0,183,100.0


## 8. Comparaison par groupe clinique

Ce tableau est le cœur de la validation exploratoire: on compare les alertes générées par le pipeline entre groupes cliniques historiques.


In [40]:
group_summary = (
    compare_df.groupby('clinical_group', dropna=False)
    .agg(
        n_cows=(COW, 'count'),
        notifs_mean=('lameness_notifs', 'mean'),
        notifs_median=('lameness_notifs', 'median'),
        starts_mean=('lameness_starts', 'mean'),
        starts_median=('lameness_starts', 'median'),
        lameness_points_mean=('lameness_points', 'mean'),
        if_points_mean=('if_anomaly_points', 'mean'),
    )
    .reset_index()
)

group_summary


,clinical_group,n_cows,notifs_mean,notifs_median,starts_mean,starts_median,lameness_points_mean,if_points_mean
0,boiteuse_stricte,5,3.400000,4.0,4.000000,4.0,38.0,191.400000
1,intermediaire,5,3.200000,4.0,4.200000,4.0,26.2,185.400000
2,saine_stricte,2,3.500000,3.5,3.500000,3.5,36.0,185.000000
3,sans_score,18,3.611111,3.5,4.055556,4.0,39.5,192.055556


## 9. Focus sur les vaches de comparaison les plus parlantes

On isole les cas explicitement mentionnés dans notre réflexion :
- boiteuses historiques fortes : `821`, `3444`
- saines stables : `2078`, `5879`


In [43]:
focus_cows = ['821', '3444', '2078', '5879']
focus_df = compare_df[compare_df[COW].isin(focus_cows)].copy()
focus_df[[
    COW, 'Statut', 'clinical_group',
    'SLS_Baseline_Jan2019', 'SLS_Midway_Mar2019',
    'lameness_notifs', 'lameness_starts', 'lameness_points',
    'if_anomaly_points', 'coverage_mean', 'coverage_min'
]].sort_values('lameness_notifs', ascending=False)


,Cow,Statut,clinical_group,SLS_Baseline_Jan2019,SLS_Midway_Mar2019,lameness_notifs,lameness_starts,lameness_points,if_anomaly_points,coverage_mean,coverage_min
10,5879,saine,saine_stricte,0.0,0.0,7,7,72,172,100.0,100.0
0,821,boiteuse_persistante,boiteuse_stricte,2.0,2.0,4,5,35,218,100.0,100.0
1,3444,boiteuse_aggravee,boiteuse_stricte,2.0,4.0,4,5,54,204,100.0,100.0
11,2078,saine,saine_stricte,0.0,0.0,0,0,0,198,100.0,100.0


## 10. Vue journalière des alertes par vache

On construit un tableau journalier utile pour vérifier si les alertes se concentrent sur quelques journées ou sont plus diffuses.


In [46]:
daily_df = (
    pred_df.assign(Day=pd.to_datetime(pred_df[TIME]).dt.floor('D'))
    .groupby([COW, 'Day'], as_index=False)
    .agg(
        day_alert=('pred_lameness_episode', 'max'),
        alert_rate=('pred_lameness_episode', 'mean'),
        day_notifs=('notif_lameness', 'sum'),
        n_bins=('pred_lameness_episode', 'size'),
        if_points=('if_anomaly_point', 'sum'),
    )
)

daily_df.head(20)


,Cow,Day,day_alert,alert_rate,day_notifs,n_bins,if_points
0,2041,2019-11-11,0,0.000000,0,21,3
1,2041,2019-11-12,1,0.062500,1,96,10
2,2041,2019-11-13,0,0.000000,0,96,3
3,2041,2019-11-14,0,0.000000,0,96,8
4,2041,2019-11-15,1,0.312500,1,96,16
5,2041,2019-11-16,0,0.000000,0,96,4
6,2041,2019-11-17,0,0.000000,0,96,2
7,2041,2019-11-18,1,0.010417,1,96,7
8,2041,2019-11-19,0,0.000000,0,96,6
9,2041,2019-11-20,0,0.000000,0,96,6


## 11. Exports CSV

Le notebook exporte quatre fichiers :
- `mcgill_pipeline_summary.csv` : résumé 1 ligne/vache
- `mcgill_pipeline_predictions.csv` : prédictions détaillées 1 ligne/bin
- `mcgill_pipeline_daily_alerts.csv` : résumé journalier
- `mcgill_pipeline_vs_sls.csv` : résumé fusionné avec les labels SLS


In [49]:
summary_df.to_csv(SUMMARY_OUT, index=False)
pred_df.to_csv(PRED_OUT, index=False)
daily_df.to_csv(DAILY_OUT, index=False)
compare_df.to_csv(COMPARE_OUT, index=False)

print('✅ Exports réalisés :')
print(' -', SUMMARY_OUT)
print(' -', PRED_OUT)
print(' -', DAILY_OUT)
print(' -', COMPARE_OUT)


✅ Exports réalisés :
 - /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/mcgill_pipeline_summary.csv
 - /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/mcgill_pipeline_predictions.csv
 - /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/mcgill_pipeline_daily_alerts.csv
 - /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/mcgill_pipeline_vs_sls.csv


## 12. Interprétation honnête

Si les vaches `boiteuse_stricte` ont en moyenne plus d'alertes que les `saine_stricte`, cela fournit un **indice externe encourageant**. Mais comme les labels cliniques et les capteurs ne sont pas synchrones, ce résultat doit être formulé comme une **validation exploratoire externe partielle**, pas comme une preuve clinique.
